In [ ]:

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'


In [ ]:

!pip install --no-deps tensorflow tf_keras tensorflow-text


In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import pandas as pd

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [ ]:
df = pd.read_csv("/content/spam.csv")
df.head(5)

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df['Category'].value_counts()

,count
Category,
ham,4825
spam,747


We are facing the class imbalance problem here.

In [ ]:
747/4825

0.15481865284974095

the result implies that about 15% are spam emails and 85% are ham emails.

In [ ]:
df_spam = df[df['Category']=='spam']
df_spam.shape

(747, 2)

In [ ]:
df_ham = df[df['Category']=='ham']
df_ham.shape

(4825, 2)

In [ ]:
df_ham_downsampled = df_ham.sample(df_spam.shape[0])
df_ham_downsampled.shape

(747, 2)

In [ ]:
df_balanced = pd.concat([df_ham_downsampled, df_spam])
df_balanced.shape

(1494, 2)

In [ ]:
df_balanced['Category'].value_counts()

,count
Category,
ham,747
spam,747


In [ ]:
df_balanced['spam'] = df_balanced['Category'].apply(lambda x: 1 if x=='spam' else 0)
df_balanced.sample(5)

,Category,Message,spam
273,spam,HMV BONUS SPECIAL 500 pounds of genuine HMV vo...,1
1849,ham,I dont want to hear philosophy. Just say what ...,0
1269,spam,Can U get 2 phone NOW? I wanna chat 2 set up m...,1
2929,ham,Anything...,0
4860,ham,"Hey, a guy I know is breathing down my neck to...",0


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df_balanced['Message'], df_balanced['spam'], stratify=df_balanced['spam'])

In [ ]:
X_train.head(4)

,Message
1982,"Sorry, I'll call later in meeting any thing re..."
4329,Congrats! 2 mobile 3G Videophones R yours. cal...
86,For real when u getting on yo? I only need 2 m...
139,You'll not rcv any more msgs from the chat svc...


In [ ]:
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4")

#bert_preprocessor_url = "https://kaggle.com/models/tensorflow/bert/TensorFlow2/en-uncased-preprocess/3"
bert_preprocessor = hub.KerasLayer(bert_preprocess, name='bert_preprocessing')



In [ ]:
class Preprocess_Layer(tf.keras.layers.Layer):
    def call(self, inputs):
        return bert_preprocessor(inputs)

In [ ]:

class Bert_Embedding_Layer(tf.keras.layers.Layer):
    def call(self, inputs):
        return bert_encoder(inputs)

In [ ]:
# Bert layers
text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessed_text = Preprocess_Layer()(text_input)
outputs = Bert_Embedding_Layer()(preprocessed_text)

# Neural network layers
l = tf.keras.layers.Dropout(0.1, name="dropout")(outputs['pooled_output'])
l = tf.keras.layers.Dense(1, activation='sigmoid', name="output")(l)

# Use inputs and outputs to construct a final model
model = tf.keras.Model(inputs=[text_input], outputs = [l])

In [ ]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text (InputLayer)           [(None,)]                    0         []                            
                                                                                                  
 preprocess__layer (Preproc  {'input_type_ids': (None,    0         ['text[0][0]']                
 ess_Layer)                  128),                                                                
                              'input_mask': (None, 128)                                           
                             , 'input_word_ids': (None,                                           
                              128)}                                                               
                                                                                              

In [ ]:
len(X_train)

1120

In [ ]:
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=METRICS)

In [ ]:
model.fit(X_train, y_train, epochs=10)

Epoch 1/10
35/35 [==============================] - 476s 13s/step - loss: 0.6210 - accuracy: 0.6580 - precision: 0.6421 - recall: 0.7143
Epoch 2/10
35/35 [==============================] - 431s 12s/step - loss: 0.5149 - accuracy: 0.7821 - precision: 0.7904 - recall: 0.7679
Epoch 3/10
35/35 [==============================] - 399s 11s/step - loss: 0.4548 - accuracy: 0.8277 - precision: 0.8148 - recall: 0.8482
Epoch 4/10
35/35 [==============================] - 401s 11s/step - loss: 0.4076 - accuracy: 0.8652 - precision: 0.8484 - recall: 0.8893
Epoch 5/10
35/35 [==============================] - 397s 11s/step - loss: 0.3783 - accuracy: 0.8679 - precision: 0.8564 - recall: 0.8839
Epoch 6/10
35/35 [==============================] - 405s 12s/step - loss: 0.3568 - accuracy: 0.8777 - precision: 0.8591 - recall: 0.9036
Epoch 7/10
35/35 [==============================] - 408s 12s/step - loss: 0.3329 - accuracy: 0.8857 - precision: 0.8763 - recall: 0.8982
Epoch 8/10
35/35 [=======================

In [ ]:
model.evaluate(X_test, y_test)

12/12 [==============================] - 143s 12s/step - loss: 0.2887 - accuracy: 0.8957 - precision: 0.8592 - recall: 0.9465


[0.28868404030799866,
 0.8957219123840332,
 0.8592233061790466,
 0.9465240836143494]

In [ ]:
y_predicted = model.predict(X_test)

y_predicted = y_predicted.flatten()

12/12 [==============================] - 140s 12s/step


In [ ]:
import numpy as np

y_predicted = np.where(y_predicted > 0.5, 1, 0)

y_predicted

array([1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0,
       1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1,
       0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1,
       1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_predicted)
cm

array([[158,  29],
       [ 10, 177]])

In [ ]:
print(classification_report(y_test, y_predicted))

              precision    recall  f1-score   support

           0       0.94      0.84      0.89       187
           1       0.86      0.95      0.90       187

    accuracy                           0.90       374
   macro avg       0.90      0.90      0.90       374
weighted avg       0.90      0.90      0.90       374



In [ ]:
reviews = [
    'Enter a chance to win $5000, hurry up, offer valid until march 31, 2021',
    'You are awarded a SiPix Digital Camera! call 09061221061 from landline.',
    'Hey Sam, Are you coming for a cricket game tomorrow',
    "Why don't you wait 'til at least wednesday to see if you get your."
]

model.predict(reviews)

1/1 [==============================] - 3s 3s/step


array([[0.80057365],
       [0.92707634],
       [0.2673462 ],
       [0.13437256]], dtype=float32)

In this tutorial, we learned how to build a spam detection model. The model was able to classify email messages as spam or ham. We started by using BERT to convert a given sentence into an embedding vector. This was done using the pretrained BERT models.

We created our model using Tensorflow and initialized all the input and output layers. We followed all the stages of building the neural network and finally came up with a spam detection model. Finally, we used the model to make predictions, the model was able to give accurate predictions.